In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 04_gold/04_dim_provider
# Purpose  : SCD Type 1 via dlt.apply_changes + Join with dim_organization
#            to retrieve the surrogate organization_key instead of raw organization_id.
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.view(name="v_provider_changes")
def v_provider_changes():
    orgs = spark.read.table("covid19_lakehouse.gold.dim_organization").select(
        F.col("organization_id"), F.col("organization_key")
    )
    providers = dlt.read_stream("covid19_lakehouse.silver.providers")
    return (
        providers
        .join(orgs, on="organization_id", how="left")
        .select(
            F.xxhash64("provider_id").alias("provider_key"),
            "provider_id",
            "provider_name",
            "gender",
            "speciality",
            "organization_key",
            "address",
            "city",
            "state",
            "zip",
            "lat",
            "lon",
            "utilization",
            "_ingested_at",
        )
    )


dlt.create_streaming_table("covid19_lakehouse.gold.dim_provider")

dlt.apply_changes(
    target="covid19_lakehouse.gold.dim_provider",
    source="v_provider_changes",
    keys=["provider_id"],
    sequence_by=F.col("_ingested_at"),
    stored_as_scd_type=1,
)